This notebook provides a benchmarking of the surrogate + SAEM implementation on a well-known data set, the **theophylline** data.

The study was reported by Upton and analyzed by Boeckmann et al. (1994).


In [ ]:
import pandas as pd 
import torch
from plotnine import *

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
# Setup the training data for the surrogate model
# An analytical expression is available for the model


def analytical_model(d, v, ka, cl, t):
    """Analytical expression of a 1 compartment PK model

    Args:
        t: time in h
        d: Dose in mg
        v: Distribution volume in mL
        ka: Absorption rate constant in mL/h
        cl: Clearance rate constant in mL/h

    Returns:
        y: Predicted concentration
    """
    ke = cl / v
    y = d * ka / (v * (ka - ke)) * (torch.exp(-ke * t) - torch.exp(-ka * t))
    return y


struct_model = StructuralAnalytical(analytical_model, ["concentration"])

In [ ]:
theophylline_data_url = "https://monolixsuite.slp-software.com/__attachments/a_2acdcb24e3aa515c4da35ea2dd93e97ce8ef9661c6086afb40937ba697a30bca/theophylline_data.txt?cb=bf00f82a5318f623eea61d62202d9b0e"

df = pd.read_csv(theophylline_data_url, sep="\t")

display(df.head())

patients_df = df.loc[df["AMT"] != "."].copy()
patients_df.loc[:, "d"] = patients_df.apply(
    lambda r: float(r["AMT"]) * r["WEIGHT"], axis=1
)
patients_df = patients_df.rename(columns={"ID": "id", "WEIGHT": "bw"})[
    ["id", "d", "bw"]
]
display(patients_df.head())
obs_df = (
    df.loc[df["AMT"] == "."]
    .copy()
    .rename(columns={"ID": "id", "CONC": "value", "TIME": "time"})[
        ["id", "time", "value"]
    ]
    .astype({"value": "float"})
)
obs_df["output_name"] = "concentration"
obs_df["protocol_arm"] = "identity"
obs_df = obs_df.merge(patients_df, on=["id"])
display(obs_df.head())

In [ ]:
model_params = {
    "pdu": {
        "ka": {"prior": 0.1, "prior_omega": 0.5},
        "cl": {
            "prior": 0.1,
            "prior_omega": 0.5,
            "covariates": {"bw": {"coef_name": "cov_bw_cl", "prior": 0.1}},
        },
        "v": {"prior": 1.0, "prior_omega": 0.5},
    },
    "pdk": ["d"],
    "error_model": {"concentration": {"error_type": "additive", "sigma": 1.0}},
}
config = Config(
    nlme=NlmeConfigDict(nb_chains=5),
    saem=SaemConfigDict(nb_phase1_iterations=200, nb_phase2_iterations=200),
)
nlme_model = NlmeModel(
    structural_model=struct_model, df=obs_df, prior_params=model_params, config=config
)

nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(1000)
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.weighted_residuals("pwres")

In [ ]:
nlme_model.plot.weighted_residuals("npde")

In [ ]:
nlme_model.plot.map_vs_posterior(3)

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.diagnostics.compute_shrinkage()